# Fair Loan AI — Bias Exploration Notebook
Run this notebook to explore bias in Indian credit scoring models interactively.

In [ ]:
import sys
sys.path.append('../backend')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from data_generator import generate_synthetic_data
from audit_engine import run_audit

# Generate synthetic data
df = generate_synthetic_data(n_samples=5000, seed=42)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Approval rates by gender
gender_rates = df.groupby('gender')['model_approved'].mean()
print('Approval rates by gender:')
print(gender_rates.to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Approval Rates by Demographic Group', fontsize=14, fontweight='bold')

# Gender
gender_rates.plot(kind='bar', ax=axes[0], color=['#4a90d9', '#e94560', '#7ec8e3'])
axes[0].set_title('By Gender'); axes[0].set_ylabel('Approval Rate')
axes[0].axhline(0.8 * gender_rates['Male'], color='red', linestyle='--', label='4/5 threshold')
axes[0].legend(); axes[0].set_xticklabels(gender_rates.index, rotation=0)

# Religion
rel_rates = df.groupby('religion')['model_approved'].mean().sort_values()
rel_rates.plot(kind='barh', ax=axes[1], color='#ffd166')
axes[1].set_title('By Religion')
axes[1].axvline(0.8 * df[df['religion']=='Hindu']['model_approved'].mean(), color='red', linestyle='--')

# City Tier
tier_rates = df.groupby('city_tier')['model_approved'].mean()
tier_rates.plot(kind='bar', ax=axes[2], color=['#06d6a0', '#ffd166', '#ff6b6b'])
axes[2].set_title('By City Tier'); axes[2].set_xticklabels(['Tier 1', 'Tier 2', 'Tier 3'], rotation=0)

plt.tight_layout()
plt.savefig('approval_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Run full audit
report = run_audit(df, model_type='demo')

print(f"Audit ID: {report['audit_id']}")
print(f"Risk Score: {report['risk_score']}/100 ({report['risk_level']})")
print(f"RBI Compliant: {report['rbi_compliant']}")
print()

for attr, data in report['bias_analysis'].items():
    print(f"{attr.upper()} — {data['severity']}")
    print(f"  {data['summary']}")
    print()

In [ ]:
# Disparate Impact heatmap
import pandas as pd

heatmap_data = {}
for attr, data in report['bias_analysis'].items():
    di = data['disparate_impact']
    heatmap_data[attr] = {g: v['di_ratio'] for g, v in di.items()}

hm_df = pd.DataFrame(heatmap_data).T

fig, ax = plt.subplots(figsize=(10, 4))
import matplotlib.colors as mcolors
cmap = plt.cm.RdYlGn
im = ax.imshow(hm_df.values.astype(float), cmap=cmap, vmin=0.5, vmax=1.1, aspect='auto')

ax.set_xticks(range(len(hm_df.columns)))
ax.set_xticklabels(hm_df.columns, rotation=45, ha='right')
ax.set_yticks(range(len(hm_df.index)))
ax.set_yticklabels(hm_df.index)

for i in range(len(hm_df.index)):
    for j in range(len(hm_df.columns)):
        val = hm_df.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if val < 0.75 else 'black')

plt.colorbar(im, ax=ax, label='Disparate Impact Ratio (≥0.8 = fair)')
ax.set_title('Disparate Impact Heatmap — Red = Bias', fontsize=13, fontweight='bold')
ax.axhline(-0.5, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('di_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Mitigation suggestions
print('MITIGATION SUGGESTIONS\n' + '='*50)
for m in report['mitigation_suggestions']:
    print(f"[{m['priority']}] {m['technique']} ({m['attribute']})")
    print(f"  {m['description']}")
    if 'fairlearn_api' in m:
        print(f"  API: {m['fairlearn_api']}")
    print()